В этом ноутбуке я проверил, может ли модель посмотреть на данные, посмотреть на pydantic BaseModel и улучшить его

In [ ]:
import os
from datetime import date
from dotenv import load_dotenv
from datasets import load_from_disk
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field


load_dotenv()

In [ ]:
dataset = load_from_disk("../data/raw")

In [ ]:
class ActInfo(BaseModel):
    full_name: str | None = Field(default=None, description="Полное наименование правового акта (тип + наименование государственного органа)")
    publication_date: date | None = Field(default=None, description="Дата опубликования документа")
    number: str | None = Field(default=None, description="Номер документа (например, 3789-р)")
    title: str | None = Field(default=None, description="Заголовок документа")
    government_agency_name: str | None = Field(default=None, description="Наименование государственного органа")
    signatory: str | None = Field(default=None, description="Официальное лицо, подписавшее акт")
    type: str | None = Field(default=None, description="Тип правового акта")

Отрабатывает в .py-файле, можно программно получить pydantic класс в текстовом виде

In [ ]:
import inspect
inspect.getsource(ActInfo)

In [ ]:
refine_prompt = """
Тебе дан pydantic BaseModel класс, описыващий структуру данных. Проанализируй примеры данных посмотри, что можно улучшить в схеме, если необходимо, добавь описание и примеры.

Текущая версия класса:

class ActInfo(BaseModel):
    full_name: str | None = Field(default=None, description="Полное наименование правового акта (тип + наименование государственного органа)")
    publication_date: date | None = Field(default=None, description="Дата опубликования документа")
    number: str | None = Field(default=None, description="Номер документа (например, 3789-р)")
    title: str | None = Field(default=None, description="Заголовок документа")
    government_agency_name: str | None = Field(default=None, description="Наименование государственного органа")
    signatory: str | None = Field(default=None, description="Официальное лицо, подписавшее акт")
    type: str | None = Field(default=None, description="Тип правового акта")

Примеры данных:
{data_samples}
"""

In [ ]:
class SchemaGuidedReasoning(BaseModel):
    refined_pydantic_class: str = Field(description="Код для создания обновлённого pydantic класса")
    reasoning: str = Field(description="Какие были внесены изменения и почему")

In [ ]:
data_samples = ""

i = 0
for row in dataset["train"]:

    if len(row["text"]) > 1000:
        continue

    data_samples += f"Полный текст:\n{row["text"]}\n"
    data_samples += f"{ActInfo(**row)}\n\n"
    
    i += 1
    if i == 5:
        break

In [ ]:
llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    model_name=os.getenv("OPENAI_REFINE_MODEL"),
)

In [ ]:
response = llm.with_structured_output(SchemaGuidedReasoning).invoke(refine_prompt.format(data_samples=data_samples))

In [ ]:
print(response.model_dump()["reasoning"])

In [ ]:
print(response.model_dump()["refined_pydantic_class"])

In [ ]:
class ActInfo(BaseModel):
    """Модель для описания структуры данных правового акта."""
    
    type: str | None = Field(
        default=None,
        description="Тип правового акта (например, 'Указ', 'Постановление', 'Распоряжение')"
    )
    
    government_agency_name: str | None = Field(
        default=None,
        description="Полное наименование государственного органа, издавшего акт (например, 'Президент Российской Федерации')"
    )
    
    full_name: str | None = Field(
        default=None,
        description="Полное наименование акта (комбинация типа и органа). Избыточное поле, сохранено для совместимости"
    )
    
    publication_date: date | None = Field(
        default=None,
        description="Дата опубликования документа в формате YYYY-MM-DD"
    )
    
    number: str | None = Field(
        default=None,
        description="Номер документа с суффиксами (например, '1406', '372', '669-рп', '433-р')"
    )
    
    title: str | None = Field(
        default=None,
        description="Заголовок документа (если явно указан в тексте). Для документов без заголовка - None"
    )
    
    signatory: str | None = Field(
        default=None,
        description="ФИО подписанта в формате 'Фамилия И.' (инициал после фамилии)"
    )